In [4]:
import zipfile
import os
import xarray as xr

file = 'era5_koridor_jawabarat_jakarta_2024_08.nc'

# .nc tipu tipu ternyata aslinya ZIP
with zipfile.ZipFile(file, 'r') as zip_ref:
    # ekstrak ke dalam folder saat ini
    zip_ref.extractall('.')
    print("Ekstraksi Berhasil!")
    print("File asli telah ditemukan: 'data_0.nc'")

# cek
if os.path.exists('data_0.nc'):
    # Buka file asli hasil ekstrak tadi dengan xarray
    ds = xr.open_dataset('data_0.nc', engine='netcdf4')
    
    print("\n--- BERHASIL TERBUKA! BERIKUT STRUKTUR DATANYA ---")
    print(ds)
else:
    print("File data_0.nc tidak ditemukan, periksa kembali folder Anda.")

Ekstraksi Berhasil!
File asli telah ditemukan: 'data_0.nc'

--- BERHASIL TERBUKA! BERIKUT STRUKTUR DATANYA ---
<xarray.Dataset> Size: 2MB
Dimensions:     (valid_time: 744, latitude: 13, longitude: 14)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 6kB 2024-08-01 ... 2024-08-31T23:...
    expver      (valid_time) <U4 12kB ...
  * latitude    (latitude) float64 104B -6.0 -6.1 -6.2 -6.3 ... -7.0 -7.1 -7.2
  * longitude   (longitude) float64 112B 106.5 106.6 106.7 ... 107.6 107.7 107.8
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 542kB ...
    d2m         (valid_time, latitude, longitude) float32 542kB ...
    tp          (valid_time, latitude, longitude) float32 542kB ...
    sp          (valid_time, latitude, longitude) float32 542kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             

In [6]:
import xarray as xr
import pandas as pd

# 1. Buka file murni hasil ekstraksi
ds = xr.open_dataset('data_0.nc', engine='netcdf4')

# 2. Salin dataset untuk mempertahankan data asli jika dibutuhkan
ds_mentah = ds.copy()

# 3. Konversi seluruh satuan ke satuan standar operasional (tanpa meringkas data)
ds_mentah['Suhu_2m_C'] = ds_mentah['t2m'] - 273.15
ds_mentah['Titik_Embun_C'] = ds_mentah['d2m'] - 273.15
ds_mentah['Curah_Hujan_mm'] = ds_mentah['tp'] * 1000
ds_mentah['Tekanan_Permukaan_hPa'] = ds_mentah['sp'] / 100

# 4. Pilih variabel yang sudah dikonversi + variabel bawaan sistem ('expver', 'number') jika ada
# Kita tidak melakukan .mean() atau penyaringan apa pun di sini
ds_eksport = ds_mentah[['Suhu_2m_C', 'Titik_Embun_C', 'Curah_Hujan_mm', 'Tekanan_Permukaan_hPa']]

# 5. Ubah struktur 3D (Waktu, Lat, Lon) langsung menjadi bentuk tabel Flat (2D DataFrame)
# reset_index() memastikan valid_time, latitude, dan longitude menjadi kolom tabel biasa
df_asli = ds_eksport.to_dataframe().reset_index()

# 6. Simpan hasil baris per baris secara utuh ke format CSV
nama_csv = 'era5_data_asli_tanpa_ringkasan_agustus_2024.csv'
df_asli.to_csv(nama_csv, index=False)

print(f"Berhasil mengeksport seluruh data tanpa diringkas!")
print(f"Nama file: {nama_csv}")
print(f"Total baris data murni: {len(df_asli)} baris.")

# Tampilkan 10 baris pertama untuk memeriksa strukturnya
print("\nTampilan 10 data teratas di dalam tabel:")
print(df_asli.head(10))

Berhasil mengeksport seluruh data tanpa diringkas!
Nama file: era5_data_asli_tanpa_ringkasan_agustus_2024.csv
Total baris data murni: 135408 baris.

Tampilan 10 data teratas di dalam tabel:
  valid_time  latitude  longitude  Suhu_2m_C  Titik_Embun_C  Curah_Hujan_mm  \
0 2024-08-01      -6.0      106.5        NaN            NaN             NaN   
1 2024-08-01      -6.0      106.6        NaN            NaN             NaN   
2 2024-08-01      -6.0      106.7        NaN            NaN             NaN   
3 2024-08-01      -6.0      106.8        NaN            NaN             NaN   
4 2024-08-01      -6.0      106.9        NaN            NaN             NaN   
5 2024-08-01      -6.0      107.0        NaN            NaN             NaN   
6 2024-08-01      -6.0      107.1  25.837067      24.206696        0.156212   
7 2024-08-01      -6.0      107.2        NaN            NaN             NaN   
8 2024-08-01      -6.0      107.3        NaN            NaN             NaN   
9 2024-08-01      -6